#Homework 6

Projects in AI & ML, Spring 2026

Barbara Kotlan

4/13/2026

#Part 1: Reinforcement Learning

Task 1 (40 points): Implement value iteration or policy iteration for a small discrete Markov Decision
Process (MDP).

You may use a grid world, FrozenLake, inventory control, or another small discrete environment
approved in class.

Perform the following:
- Clearly define the state space, action space, reward structure, and discount factor.
- Implement either value iteration or policy iteration and compute the final policy.
- Show the learned value function and the final policy in a readable form.
- Run at least one small experiment showing how the policy changes when you vary either the
discount factor or the reward design.
- Briefly discuss why this setup is an MDP and what the learned policy is doing.

`#Part 1`

Using FrozenLake 4x4 grid world:

- State space (S): Each grid cell is a state (16 states: 0 to 15).
- Action space (A): {Up, Down, Left, Right} → 4 discrete actions.
- Reward structure (R):
  - +1 for reaching the goal state
  - 0 for frozen spots
  - -1 for holes
- Discount factor (γ): 0.9

Grid:
| Col 1 | Col 2 | Col 3 | Col 4 |
|---|---|---|---|
| S | F | F | F |
| F | H | F | H |
| F | F | F | H |
| H | F | F | G |

- S = Start
- F = Frozen
- H = Hole
- G = Goal

In [ ]:
import numpy as np

#Define 4x4 FrozenLake grid
#S = Start, F = Frozen, H = Hole, G = Goal
lake = [
    ['S', 'F', 'F', 'F'],
    ['F', 'H', 'F', 'H'],
    ['F', 'F', 'F', 'H'],
    ['H', 'F', 'F', 'G']
]

n_rows, n_cols = 4, 4
n_states = n_rows * n_cols
actions = ['up', 'down', 'left', 'right']
n_actions = len(actions)

#Mapping cell to reward
R = np.zeros(n_states)
for i in range(n_rows):
    for j in range(n_cols):
        s = i * n_cols + j
        if lake[i][j] == 'G':
            R[s] = 1
        elif lake[i][j] == 'H':
            R[s] = -1  #penalty for falling into a hole

#Transition probabilities: 80% intended, 10% left/right
def get_next_states_probs(state, action):
    row, col = divmod(state, n_cols)
    if lake[row][col] in ['H', 'G']:
        # terminal states
        return [(state, 1.0)]

    next_states = []
    #intended move
    drc = {
        'up': (-1, 0),
        'down': (1, 0),
        'left': (0, -1),
        'right': (0, 1)
    }
    def move(r, c, dr, dc):
        r_new = min(max(r + dr, 0), n_rows - 1)
        c_new = min(max(c + dc, 0), n_cols - 1)
        return r_new * n_cols + c_new

    #intended direction
    s_next = move(row, col, *drc[action])
    next_states.append((s_next, 0.8))

    #perpendicular moves
    if action in ['up', 'down']:
        for a in ['left', 'right']:
            s_perp = move(row, col, *drc[a])
            next_states.append((s_perp, 0.1))
    else:
        for a in ['up', 'down']:
            s_perp = move(row, col, *drc[a])
            next_states.append((s_perp, 0.1))

    return next_states

#Value Iteration function
def value_iteration(R, gamma=0.9, theta=1e-4):
    V = np.zeros(n_states)
    policy = np.zeros(n_states, dtype=int)

    while True:
        delta = 0
        for s in range(n_states):
            if lake[s//n_cols][s%n_cols] in ['H','G']:
                continue
            v = V[s]
            action_values = np.zeros(n_actions)
            for a_idx, a in enumerate(actions):
                val = 0
                for s_next, prob in get_next_states_probs(s, a):
                    val += prob * (R[s_next] + gamma * V[s_next])
                action_values[a_idx] = val
            V[s] = np.max(action_values)
            policy[s] = np.argmax(action_values)
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break
    return V, policy

#Display value function and policy
def display(V, policy):
    arrow_map = ['↑', '↓', '←', '→']
    policy_arrows = [arrow_map[a] if lake[i//n_cols][i%n_cols]=='F' else lake[i//n_cols][i%n_cols]
                     for i,a in enumerate(policy)]
    print("Value Function:")
    print(V.reshape(n_rows, n_cols).round(2))
    print("Policy:")
    print(np.array(policy_arrows).reshape(n_rows, n_cols))

#Run value iteration
V, policy = value_iteration(R, gamma=0.9)
display(V, policy)

Value Function:
[[0.18 0.12 0.21 0.1 ]
 [0.21 0.   0.26 0.  ]
 [0.41 0.68 0.64 0.  ]
 [0.   0.81 0.94 0.  ]]
Policy:
[['S' '↑' '↓' '↑']
 ['↓' 'H' '↓' 'H']
 ['→' '↓' '↓' 'H']
 ['H' '→' '→' 'G']]


In [ ]:
for gamma_test in [0.5, 0.9]:
    V, policy = value_iteration(R, gamma=gamma_test)
    print(f"\nDiscount Factor γ = {gamma_test}")
    display(V, policy)


Discount Factor γ = 0.5
Value Function:
[[ 0.    0.    0.    0.  ]
 [ 0.    0.   -0.1   0.  ]
 [ 0.01  0.16  0.25  0.  ]
 [ 0.    0.37  0.86  0.  ]]
Policy:
[['S' '↑' '↑' '↑']
 ['←' 'H' '↓' 'H']
 ['↑' '↓' '↓' 'H']
 ['H' '→' '→' 'G']]

Discount Factor γ = 0.9
Value Function:
[[0.18 0.12 0.21 0.1 ]
 [0.21 0.   0.26 0.  ]
 [0.41 0.68 0.64 0.  ]
 [0.   0.81 0.94 0.  ]]
Policy:
[['S' '↑' '↓' '↑']
 ['↓' 'H' '↓' 'H']
 ['→' '↓' '↓' 'H']
 ['H' '→' '→' 'G']]


Why this is an MDP:

The next state depends only on the current state and action (Markov property).
The environment has finite discrete states and actions.
The reward is defined for each state-action outcome, and transitions have probabilities (stochastic).

#Part 2: Mini Research Task on a Novel AI Topic

Task 2 (60 points): Complete one scoped mini research task on a recent AI topic. Diffusion models
are strongly encouraged, but you may choose another novel topic.

Examples of acceptable topics include diffusion models, consistency models, flow matching,
multimodal foundation models, retrieval-augmented generation, parameter-efficient fine-tuning,
graph neural networks, or another recent topic in the field of AI.

Perform the following:
- Select one focused topic and identify at least two technical sources. At least one source should
be a primary source such as a research paper, official documentation, or a model card.
- Explain the problem the method is trying to solve, the core idea behind the method, and the main
architecture or training objective in your own words.
- Include one small hands-on component. For example, you may run a simple demo, inspect a
pretrained model or pipeline, reproduce one small figure or result, compare two variants, or
analyze how one key design choice affects outputs.
- If you choose diffusion models, possible hands-on directions include comparing different
numbers of denoising steps, comparing schedulers, testing prompt sensitivity, or explaining the
forward and reverse diffusion process with a compact experiment.
- Include at least one figure, table, or carefully labeled output example that supports your analysis.
- Discuss at least two limitations, risks, or open challenges related to the method.
- Propose one concrete extension, improvement, or follow-up experiment that you would pursue
next.

The goal is depth on one focused recent topic, not breadth across many topics.

`#Part 2`

Topic: Diffusion Models

Sources
- https://ieeexplore.ieee.org/abstract/document/10081412

This survey paper provides a comprehensive overview of diffusion models in computer vision, highlighting their rapid rise as a state-of-the-art approach for generative modeling. It explains that diffusion models operate through a two-stage process: a forward process that gradually adds Gaussian noise to data, and a reverse process where a neural network learns to denoise and reconstruct the original data step-by-step. The paper emphasizes that diffusion models achieve very high-quality and diverse image generation results, often outperforming earlier generative methods such as GANs. It also discusses major applications including image generation, super-resolution, inpainting, and image editing, as well as emerging uses in representation learning tasks like classification and segmentation. Despite their strong performance, the paper identifies a key limitation: slow sampling speed due to the large number of iterative denoising steps required during generation. Finally, it highlights the rapid growth of the field and suggests that diffusion models will continue to expand into new applications and improvements in efficiency.

- https://papers.baulab.info/papers/also/Luo-2022.pdf

This paper presents a unified explanation of diffusion models by connecting them to other generative modeling approaches such as VAEs, GANs, and score-based models. It describes how diffusion models work by gradually adding Gaussian noise to data in a forward process and then learning to reverse this process through iterative denoising. The training objective can be derived from variational inference and is closely related to the Evidence Lower Bound (ELBO), often simplified to predicting the noise added at each step. A key contribution of the paper is showing that diffusion models can be interpreted from multiple equivalent perspectives, including likelihood-based and score-based formulations. It also discusses practical techniques such as classifier-free guidance that improve controllability during generation. Finally, it highlights limitations such as slow sampling due to many denoising steps and motivates research into more efficient generation methods.

Problem:

Diffusion models aim to address the problem of learning complex, high-dimensional data distributions in a stable and high-quality manner, particularly for tasks such as image generation. Traditional generative approaches can suffer from instability during training or produce low-diversity outputs, motivating more robust alternatives.


Core Idea:

The core idea is to reformulate generation as a denoising process, where data is gradually corrupted by adding Gaussian noise over multiple steps. A neural network is then trained to reverse this process step-by-step, learning how to recover clean data from noisy inputs. During generation, the model starts from pure noise and iteratively denoises it to produce realistic samples.


Training Objective:

Instead of directly predicting the original image, the model is trained to predict the noise that was added at each timestep. This simplifies the learning problem and leads to more stable optimization. The objective is derived from variational inference principles and is commonly expressed as a mean squared error between the true noise and the predicted noise.


$L = \mathbb{E}_{x_0, \epsilon, t} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$

In [1]:
pip install diffusers transformers accelerate torch

In [3]:
#Hands on Component
import torch
from diffusers import StableDiffusionPipeline
import time

#Load model
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5"
)
pipe = pipe.to("cpu")

prompt = "A photo of a golden retriever in a park"

steps_list = [5, 10, 20]

results = []

for steps in steps_list:
    start = time.time()

    image = pipe(prompt, num_inference_steps=steps).images[0]

    end = time.time()
    elapsed = end - start

    filename = f"output_{steps}_steps.png"
    image.save(filename)

    results.append((steps, elapsed, filename))
    print(f"Steps: {steps}, Time: {elapsed:.2f}s, Saved: {filename}")

print("\nSummary:")
for r in results:
    print(r)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Steps: 5, Time: 214.94s, Saved: output_5_steps.png


  0%|          | 0/10 [00:00<?, ?it/s]

Steps: 10, Time: 312.12s, Saved: output_10_steps.png


  0%|          | 0/20 [00:00<?, ?it/s]

Steps: 20, Time: 563.29s, Saved: output_20_steps.png

Summary:
(5, 214.93641757965088, 'output_5_steps.png')
(10, 312.1183891296387, 'output_10_steps.png')
(20, 563.2862985134125, 'output_20_steps.png')


| Denoising Steps | Time (seconds) | Observed Quality                  |
|-----------------|---------------|----------------------------------|
| 5               | 214.94        | Very noisy, unclear structure    |
| 10              | 312.12        | Some structure, still blurry     |
| 20              | 563.29        | Clearer image, recognizable      |



I evaluated the effect of the number of denoising steps on image generation quality and runtime using a pretrained diffusion model. As shown in Table 1, increasing the number of steps significantly improves image quality. With only 5 steps, the output image is highly noisy and lacks recognizable structure. At 10 steps, the model begins to capture some structure, although the image remains blurry. At 20 steps, the generated image is noticeably clearer and more realistic. However, this improvement comes at the cost of increased computation time, rising from approximately 215 seconds to over 560 seconds. This demonstrates a key tradeoff in diffusion models between generation quality and efficiency.

Limitaions/Risks:
1. Slow sampling - sampling requires many steps which is computaitonally expensive
2. Bias and misuse - trained on internet data which can inherit biase and also can be used for misleading content

Extentions:

A useful follow-up experiment would be to explore faster diffusion sampling methods, such as DDIM or consistency-based models, that aim to reduce the number of denoising steps while maintaining image quality. I would compare these methods against the standard diffusion process by generating images from the same prompt using a small number of inference steps (e.g., 5, 10, and 20) and evaluating both visual quality and runtime. This would help determine whether it is possible to significantly improve sampling efficiency without sacrificing output quality, addressing the main limitation of slow generation in diffusion models.
